In [33]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_FiLMSEModel, load_LoRASEModel, load_FiLMLoRASEModel, load_HyperNetworkSEModel, generate_files_with_nucleus
import torch
import numpy as np
import pickle
from tqdm import tqdm
from GridMLM_tokenizers import CSGridMLMTokenizer
from graph_utils import get_graph_embeddings_from_string_with_model, get_bilstm_embeddings_from_string_with_model, graph_from_string, compare_heterodata

In [34]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [35]:
device_name = 'cuda:0'
device = torch.device(device_name)

graph_model_FiLM_path = 'saved_models/FiLM/graph/graph_model.pt'
transformer_graph_FiLM_path = 'saved_models/FiLM/graph/transformer_model.pt'

graph_model_LoRA_path = 'saved_models/LoRA/graph/graph_model.pt'
transformer_graph_LoRA_path = 'saved_models/LoRA/graph/transformer_model.pt'

graph_model_FiLMLoRA_path = 'saved_models/FiLMLoRA/graph/graph_model.pt'
transformer_graph_FiLMLoRA_path = 'saved_models/FiLMLoRA/graph/transformer_model.pt'

graph_model_HyperNetwork_path = 'saved_models/HyperNetwork/graph/graph_model.pt'
transformer_graph_HyperNetwork_path = 'saved_models/HyperNetwork/graph/transformer_model.pt'

# bilstm_model_path = 'saved_models/bilstm/bilstm_model.pt'
# transformer_bilstm_path = 'saved_models/bilstm/transformer_model.pt'

In [36]:
graph_model_FiLM = load_GraphModel(graph_model_FiLM_path, device)
transformer_graph_FiLM = load_FiLMSEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_graph_FiLM_path
)
graph_model_LoRA = load_GraphModel(graph_model_LoRA_path, device)
transformer_graph_LoRA = load_LoRASEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_graph_LoRA_path
)
graph_model_FiLMLoRA = load_GraphModel(graph_model_FiLMLoRA_path, device)
transformer_graph_FiLMLoRA = load_FiLMLoRASEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_graph_FiLMLoRA_path
)
graph_model_HyperNetwork = load_GraphModel(graph_model_HyperNetwork_path, device)
transformer_graph_HyperNetwork = load_HyperNetworkSEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_graph_HyperNetwork_path
)

# bilstm_model = load_BiLSTMModel(bilstm_model_path, device)
# transformer_bilstm = load_AttnFiLMSEModel(
#     tokenizer,
#     device,
#     checkpoint_path=transformer_bilstm_path
# )

In [37]:
graph_model_FiLM.eval()
transformer_graph_FiLM.eval()

graph_model_LoRA.eval()
transformer_graph_LoRA.eval()

graph_model_FiLMLoRA.eval()
transformer_graph_FiLMLoRA.eval()

graph_model_HyperNetwork.eval()
transformer_graph_HyperNetwork.eval()
# bilstm_model.eval()
# transformer_bilstm.eval()

HyperNetworkSEModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (hyper_q): HyperGuide(
    (layer_embedding): Embedding(8, 16)
    (head_embedding): Embedding(8, 16)
    (mlp): Sequential(
      (0): Linear(in_features=160, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=128, bias=True)
      (3): GELU(approximate='none')
    )
    (gamma_proj): Linear(in_features=128, out_features=64, bias=True)
    (beta_proj): Linear(in_features=128, out_features=64, bias=True)
    (lora_A): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=512, bias=True)
    )
    (lora_B): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=512, bias=True)
    )
  )

In [38]:
datasets = {
    'gjt': {'path': 'data/gjt_test.pkl', 'dataset': None},
    # 'hook': {'path': 'data/hook_test.pkl', 'dataset': None},
    'nott': {'path': 'data/nott_test.pkl', 'dataset': None},
    # 'wiki': {'path': 'data/wiki_test.pkl', 'dataset': None}
}

for k, v in datasets.items():
    print(f'loading {k}')
    with open(v['path'], 'rb') as f:
        d = pickle.load(f)
    v['dataset'] = d

loading gjt
loading nott


In [39]:
# in_seq = 'b_A#:7_@2;A:min6_@2'
# in_seq = 'b_F#:7_@2;B:7_@2b_E:7_@2;A#:7_@2;'
# in_seq = 'b_A#:7_@2;A:min6_@2'
# in_seq = 'b_F:min_@2;A#:min_@2'
in_seq = 'E:min;A:maj;F:min;A#:maj'
# in_seq = 'G:maj;D:maj;G#:maj;D#:maj'

In [40]:
y_graph_FiLM = get_graph_embeddings_from_string_with_model(in_seq, graph_model_FiLM).unsqueeze(0)
y_graph_LoRA = get_graph_embeddings_from_string_with_model(in_seq, graph_model_LoRA).unsqueeze(0)
y_graph_FiLMLoRA = get_graph_embeddings_from_string_with_model(in_seq, graph_model_FiLMLoRA).unsqueeze(0)
y_graph_HyperNetwork = get_graph_embeddings_from_string_with_model(in_seq, graph_model_HyperNetwork).unsqueeze(0)

# y_bilstm = get_bilstm_embeddings_from_string_with_model(in_seq, bilstm_model)

E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297
E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297
E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297
E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297


In [41]:
# print(y_graph.unsqueeze(0).shape)
# print(y_bilstm.shape)

In [42]:
# input_f_path = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_test/Autumn_Leaves.mxl'
input_f_path = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_test/So_What.mxl'

piece_name = input_f_path.split('/')[-1].split('.')[0]
print(piece_name)

So_What


In [43]:
gen_out = generate_files_with_nucleus(
    transformer_graph_FiLM,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_no',
    guidance_vec = None,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'D#:min', 'D#:min', 'E:min', 'E:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'F#:min6', 'F#:min6', 'F#:min6', 'F#:min6', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'D#:maj', 'D#:maj', 'D#:maj', 'D#:maj', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'D#:maj', 'D#:maj', 'G:min', 'G:min', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'A#:maj', 'A#:maj', 'A#:maj', 'A#:maj']


In [44]:
# gen_out = generate_files_with_nucleus(
#     transformer_bilstm,
#     tokenizer,
#     input_f_path=input_f_path,
#     mxl_folder_out=None,
#     midi_folder_out='midi_tests',
#     name_suffix=f'{piece_name}_bilstm',
#     guidance_vec = y_bilstm,
#     use_constraints=False,
#     intertwine_bar_info=True,
#     normalize_tonality=True,
#     temperature=1.0,
#     p=0.9,
#     unmasking_order='certain',
#     create_gen=True,
#     create_real=False
# )
# print(gen_out['gen_output_tokens'])

In [45]:
gen_out = generate_files_with_nucleus(
    transformer_graph_FiLM,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_graph_FiLM',
    guidance_vec = y_graph_FiLM,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'D:maj', 'D:maj', 'D:maj', 'D:maj', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'D:maj', 'D:maj', 'D:maj', 'D:maj', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'D#:maj7', 'D#:maj7', 'D#:maj7', 'D#:maj7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'D#:maj7', 'D#:maj7', 'D#:maj7', 'D#:maj7', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'D#:maj7', 'D#:maj7', 'D#:maj7', 'D#:maj7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'A#:maj', 'A#:maj', 'A#:maj', 'A#:maj']


In [46]:
gen_out = generate_files_with_nucleus(
    transformer_graph_LoRA,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_graph_LoRA',
    guidance_vec = y_graph_LoRA,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'A:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'A:sus2', 'A:sus2', 'A:sus2', 'A:sus2', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'D#:maj', 'D#:maj', 'D#:maj', 'D#:maj', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'A#:maj7', 'A#:maj7', 'D#:maj7', 'D#:maj7', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'A#:maj', 'A#:maj', 'A#:maj', 'A#:maj', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'G:min7', 'A#:maj', 'D:7', 'D:7']


In [47]:
gen_out = generate_files_with_nucleus(
    transformer_graph_FiLMLoRA,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_graph_FiLMLoRA',
    guidance_vec = y_graph_FiLMLoRA,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'C:maj', 'C:maj', 'C:maj', 'C:maj', '<bar>', 'A:maj', 'A:maj', 'A:maj', 'A:maj', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'A#:maj', 'A#:maj', 'A#:maj', 'A#:maj', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'A#:maj', 'A#:maj', 'A#:maj', 'A#:maj', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'A#:maj', 'A#:maj', 'A#:maj', 'A#:maj', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'A#:maj', 'A#:maj', 'A#:maj', 'A#:maj']


In [48]:
gen_out = generate_files_with_nucleus(
    transformer_graph_HyperNetwork,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_graph_HyperNetwork',
    guidance_vec = y_graph_HyperNetwork,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'C:maj7', 'C:maj7', 'C:maj7', 'C:maj7', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'C:maj7', 'C:maj7', 'C:maj7', 'C:maj7', '<bar>', 'C:maj7', 'C:maj7', 'C:maj7', 'C:maj7', '<bar>', 'C:maj7', 'C:maj7', 'C:maj7', 'C:maj7', '<bar>', 'F:maj7', 'F:maj7', 'F:maj7', 'F:maj7', '<bar>', 'E:min7', 'E:min7', 'E:min7', 'E:min7', '<bar>', 'A:maj7', 'A:maj7', 'A:maj7', 'A:maj7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'D#:maj7', 'D#:maj7', 'D#:maj7', 'D#:maj7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'D#:maj7', 'D#:maj7', 'D#:maj7', 'D#:maj7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'D#:maj7', 'D#:maj7', 'D#:maj7', 'D#:maj7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'A#:maj', 'A#:maj', 'A#:maj', 'A#:maj']
